In [40]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd

In [41]:
## Load the trained model, scaler pickle file, one hot encoding pickle file
model = load_model('my_model.keras')

## load the encoders and scalers
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

/Users/arpitgupta/Documents/Deep_Learning_NLP/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [42]:
import tensorflow as tf
import keras

print(tf.__version__)
print(keras.__version__)

2.21.0
3.13.2


In [71]:
# Example input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000,
}

In [72]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [73]:
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
print(onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded

['Geography_France' 'Geography_Germany' 'Geography_Spain']


array([[1., 0., 0.]])

In [74]:
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))


In [75]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [60]:
'''
3. Pipeline mindset
Your flow should always be:
dict → DataFrame → transform → concat → reorder → scale → predict
'''

'\n3. Pipeline mindset\nYour flow should always be:\ndict → DataFrame → transform → concat → reorder → scale → predict\n'

In [61]:
'''
Use this one ✅:

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
❌ Why the other option is wrong
input_df['Gender'] = label_encoder_gender.transform([input_df['Gender']])
What you're actually passing:
[input_df['Gender']]

This becomes:

[ Series(['Male']) ]

👉 So it's a list containing a Series, not a 1D array of values.

💥 Result:

Shape becomes weird (2D-like / nested)

Can throw errors like:

ValueError: bad input shape

Or worse: silently incorrect transformation

🧠 What LabelEncoder expects
transform(X)

Where X should be:

1D array-like → ['Male']

OR pandas Series → input_df['Gender']

✔️ Your correct input:

input_df['Gender']
🔍 Mental Model
Input Type	Example	Works?
List of values	['Male']	✅
Pandas Series	input_df['Gender']	✅✅
List of Series	[input_df['Gender']]	❌
🔥 Best Practice (Always follow this)

👉 Once you have a DataFrame:

input_df = pd.DataFrame([input_data])

Then ALWAYS:

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
✅ Final Answer

✔️ Use:

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

❌ Never use:

transform([input_df['Gender']])
'''

"\nUse this one ✅:\n\ninput_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])\n❌ Why the other option is wrong\ninput_df['Gender'] = label_encoder_gender.transform([input_df['Gender']])\nWhat you're actually passing:\n[input_df['Gender']]\n\nThis becomes:\n\n[ Series(['Male']) ]\n\n👉 So it's a list containing a Series, not a 1D array of values.\n\n💥 Result:\n\nShape becomes weird (2D-like / nested)\n\nCan throw errors like:\n\nValueError: bad input shape\n\nOr worse: silently incorrect transformation\n\n🧠 What LabelEncoder expects\ntransform(X)\n\nWhere X should be:\n\n1D array-like → ['Male']\n\nOR pandas Series → input_df['Gender']\n\n✔️ Your correct input:\n\ninput_df['Gender']\n🔍 Mental Model\nInput Type\tExample\tWorks?\nList of values\t['Male']\t✅\nPandas Series\tinput_df['Gender']\t✅✅\nList of Series\t[input_df['Gender']]\t❌\n🔥 Best Practice (Always follow this)\n\n👉 Once you have a DataFrame:\n\ninput_df = pd.DataFrame([input_data])\n\nThen ALWAYS:\n\ninput_df['

In [76]:
input_df = input_df.drop('Geography', axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,1,40,3,60000,2,1,1,50000


In [77]:
input_df = pd.concat([input_df, geo_encoded_df], axis=1)

In [78]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [80]:
# Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [81]:
# prediction
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


array([[0.03415552]], dtype=float32)

In [82]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.03415552)

In [83]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn')

The customer is not likely to churn


🔹 3. Summary (VERY IMPORTANT)

1. Encoder Type	Expected Input Shape	Example Input
2. LabelEncoder	1D	['Male']
3. OneHotEncoder	2D	[['France']]
4. Scaler	2D	[[600, 40, ...]]

In [ ]:
'''

🧠 Core Design Idea of sklearn
Everything is built around this:
X → (n_samples, n_features)
Even if:
1 feature → still (n_samples, 1)
1 sample → still (1, n_features)

🔹 LabelEncoder vs OneHotEncoder (real difference)
🔸 LabelEncoder
['Male', 'Female', 'Male']

Works on target-like vector

Designed for y, not X

So it accepts 1D

🔸 OneHotEncoder
[
 ['France'],
 ['Germany'],
 ['Spain']
]

Works on feature matrix (X)

Can handle multiple columns at once

🔥 Important Insight (Most people miss this)
👉 Yes — OHE can take MULTIPLE columns at once
onehot_encoder.fit([
    ['France', 'Male'],
    ['Germany', 'Female']
])
Shape:
(2 samples, 2 features)

So sklearn enforces:
“Always give me a 2D matrix — I don’t care if it’s 1 column or 10”

🚨 Why NOT allow 1D for OHE?
Because this becomes ambiguous:
['France', 'Germany', 'Spain']
Is this:
❓ 3 samples, 1 feature → correct
❓ 1 sample, 3 features → also possible
👉 To avoid this ambiguity → sklearn forces 2D

🧩 2. The Mental Model (KEEP THIS)
Concept	Shape	Example
1 sample	(1, n_features)	[[600, 40]]
1 feature	(n_samples, 1)	[['France'], ['India']]
Target (y)	(n_samples,)	['Yes', 'No']
📚 3. MASTER TABLE (Save this 💯)

You can literally copy this into your notes.

🔹 ENCODING TECHNIQUES
Technique	Input Type	Shape Expected	Example
LabelEncoder	1D	(n_samples,)	['Male', 'Female']
OneHotEncoder	2D	(n_samples, n_features)	[['France'], ['India']]
OrdinalEncoder	2D	(n_samples, n_features)	[['Low'], ['High']]
TargetEncoder*	1D/2D	depends	varies
BinaryEncoder*	2D	(n_samples, n_features)	categorical columns

(*from category_encoders lib)

🔹 SCALING / TRANSFORMERS
Technique	Input Type	Shape Expected	Example
StandardScaler	2D	(n_samples, n_features)	[[600, 40]]
MinMaxScaler	2D	(n_samples, n_features)	same
RobustScaler	2D	(n_samples, n_features)	same
PowerTransformer	2D	(n_samples, n_features)	same
🔹 IMPUTERS
Technique	Input Type	Shape Expected
SimpleImputer	2D	(n_samples, n_features)
KNNImputer	2D	(n_samples, n_features)
🔹 FEATURE SELECTION
Technique	Input Type
SelectKBest	X → 2D, y → 1D
RFE	X → 2D, y → 1D
🔹 MODELS (VERY IMPORTANT)
Model Type	X Shape	y Shape
sklearn models	2D	1D
TensorFlow / Keras	2D	1D or 2D
🚀 4. Golden Rules (Interview-level clarity)
✅ Rule 1:

If it works on X → always 2D

✅ Rule 2:

If it works on y → usually 1D

✅ Rule 3:

If confused → wrap in DataFrame

pd.DataFrame([input_data])
🔥 5. Final Intuition (THIS is the real answer)

LabelEncoder = vector processor

OneHotEncoder = matrix processor

That’s it.

⚠️ Bonus (Advanced Insight)

LabelEncoder is actually NOT recommended for features anymore.

👉 Use instead:

OneHotEncoder (nominal)

OrdinalEncoder (ordinal)

'''

In [ ]:
'''
🧠 Short Answer
👉 Yes — OneHotEncoder can remember column names, but only if you give it that information during fit()
🔍 What actually happens internally

When you do:

onehot_encoder_geo.fit(data[['Geography']])

You're passing a DataFrame, not a NumPy array.

✔️ Because of that:

sklearn extracts and stores:

onehot_encoder_geo.feature_names_in_

This will be:

array(['Geography'], dtype=object)

So yes — it remembers the column name.

⚠️ But here's the important catch

If you instead did:

onehot_encoder_geo.fit(data['Geography'].values.reshape(-1,1))

or:

onehot_encoder_geo.fit(np.array(...))

Then:

❌ No column names are stored
❌ feature_names_in_ will NOT exist

Because NumPy arrays don’t carry column metadata
'''